# Many sandboxes in parallel

This notebook creates 20 sandboxes at once, runs a test in each and deletes them again, with `AsyncSandboxClient`. A preset's sandboxes start from its snapshot, so each step takes seconds. Each step is a cell of its own: keep the **Sandboxes** page of the prokube UI open next to the notebook and watch them appear, run and disappear. See the [README](./README.md) for how many sandboxes can run at the same time.

Sandboxes must be enabled on your prokube cluster, and the Lab needs sandbox access.

## Setup

In [ ]:
%pip install -q "prokube-sdk @ git+https://github.com/prokube/prokube-sdk@sandboxv3/main"

## The test

Each sandbox checks one candidate implementation of a function, as in an eval or a test matrix. Every fourth candidate has a bug:

In [ ]:
N = 20
PRESET = "pk-default-small"

CANDIDATES = [f"def add(a, b):\n    return a + b{' + 1' if i % 4 == 3 else ''}\n" for i in range(N)]
TEST = "assert add(2, 3) == 5, f'add(2, 3) returned {add(2, 3)}'\n"

## Create

In a Lab with sandbox access, `AsyncSandboxClient()` needs no configuration: it uses the Lab's workspace and identity. A new sandbox is suspended and takes no room, so all of them are created at once:

In [ ]:
import asyncio
import time

from prokube.sandbox import AsyncSandboxClient

sandboxes = AsyncSandboxClient()
names = [f"parallel-{i}" for i in range(N)]

start = time.perf_counter()
await asyncio.gather(*(sandboxes.create(name, preset=PRESET) for name in names))
print(f"{N} sandboxes created in {time.perf_counter() - start:.1f} s")

The **Sandboxes** page now lists them, suspended.

## Run

Running sandboxes need CPU and memory in the worker pool, so at most `RUNNING` of them run at the same time. Each one is resumed for its test and suspended again:

In [ ]:
RUNNING = 4
room = asyncio.Semaphore(RUNNING)


async def evaluate(name, candidate):
    async with room:
        await sandboxes.resume(name)
        try:
            result = await sandboxes.run(name, "python3 -", stdin=candidate + TEST)
        finally:
            await sandboxes.suspend(name)
    return result.exit_code == 0


start = time.perf_counter()
passed = await asyncio.gather(*(evaluate(name, c) for name, c in zip(names, CANDIDATES)))
print(f"{N} tests run in {time.perf_counter() - start:.1f} s")
print("failed:", [i for i, ok in enumerate(passed) if not ok])

## Clean up

Delete the sandboxes:

In [ ]:
start = time.perf_counter()
await asyncio.gather(*(sandboxes.delete(name) for name in names))
await sandboxes.close()
print(f"{N} sandboxes deleted in {time.perf_counter() - start:.1f} s")